# NSGA-NET MODEL CLASSIFIACTION

## 1. Get and prepare data

In [1]:
import os
import shutil
import random

In [ ]:
SOURCE_DIR = 'Data/images'
TRAIN_DIR = 'Data/train'
TEST_DIR = 'Data/test'
TEST_RATIO = 0.2
DATA_ENTRY_CSV = "Data_Entry_2017.csv"

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# Get all images and shuffle them
images = [f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(('.png'))]
random.shuffle(images)

# Split the images into training and testing sets
split_idx = int(len(images) * (1 - TEST_RATIO))
train_images = images[:split_idx]
test_images = images[split_idx:]

# Move files
for img in train_images:
    shutil.move(os.path.join(SOURCE_DIR,img), os.path.join(TRAIN_DIR,img))

for img in test_images:
    shutil.move(os.path.join(SOURCE_DIR,img), os.path.join(TEST_DIR,img))

print(f"Train images: {len(train_images)} | Test images: {len(test_images)}")

Train images: 0 | Test images: 0


### 1.2 Transform the data and create `datasets`

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np


In [4]:
# Create CustomDataset class
class CustomDatset(Dataset):
    """Dataset for loading images and labels from a list of dictionaries."""
    
    def __init__(self, data, classes ,transform=None):
        self.data = data
        self.transform = transform
        self.classes = classes
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = Image.open(item['img']).convert('L')
        img_array = np.array(img).astype(np.float32)
        labels = item['labels'].split('|')
        target = torch.zeros(len(self.classes))
        for label in labels:
            if label in self.classes:
                target[self.classes.index(label)] = 1
        
        if self.transform:
            img_array = self.transform(img_array)
        return img_array,target


In [ ]:
# Get the list of classes
def get_classes(data: list) -> list:
    """Extract unique classes from the dataset."""
    classes = set()
    for item in data:
        labels = item.split('|')
        classes.update(labels)
    return sorted(classes)

entry_dataframe = pd.read_csv(DATA_ENTRY_CSV)
classes = get_classes(entry_dataframe['Finding Labels'].to_list())


In [6]:
# Define the transformations for training and testing datasets
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

BATCH_SIZE = 8
train_data = []
test_data = []

for filename in os.listdir(TRAIN_DIR):
    if filename.endswith('.png'):
        img_path = os.path.join(TRAIN_DIR, filename)
        labels = entry_dataframe[entry_dataframe['Image Index'] == filename]['Finding Labels'].values
        sample = {
            'img': img_path,
            'labels': labels[0] if len(labels) > 0 else None
        }
        train_data.append(sample)

for filename in os.listdir(TEST_DIR):
    if filename.endswith('.png'):
        img_path = os.path.join(TEST_DIR, filename)
        labels = entry_dataframe[entry_dataframe['Image Index'] == filename]['Finding Labels'].values
        sample = {
            'img': img_path,
            'labels': labels[0] if len(labels) > 0 else None
        }
        test_data.append(sample)

# Create datasets
train_dataset = CustomDatset(train_data, classes=classes, transform=train_transform)
test_dataset = CustomDatset(test_data, classes=classes, transform=test_transform)

img, target = train_dataset[0]
print(f"Image shape: {img.shape} | Target shape: {target.shape}")

# Turn datasets into DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Train DataLoader size: {len(train_loader)} | Test DataLoader size: {len(test_loader)}")


Image shape: torch.Size([1, 1024, 1024]) | Target shape: torch.Size([15])
Train DataLoader size: 500 | Test DataLoader size: 125


In [7]:
# Check if all is okay
#img, target = next(iter(train_loader))
#print(f"Image shape: {img.shape} | Target shape: {target.shape}")

#img,target = next(iter(test_loader))
#print(f"Image shape: {img.shape} | Target shape: {target.shape}")

## 2. Setup model, loss function and optimizer

In [8]:
import timm
import torch
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

c:\Users\Komputer\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
model = timm.create_model('nasnetalarge',pretrained=True, num_classes=len(classes),in_chans=1)
optimizer = torch.optim.Adam(params=model.parameters(),lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

## 3. Train Model

In [ ]:
EPOCHS = 100
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

for epoch in range(EPOCHS):
  model.train()
  train_loss = 0.0
  for img, target in tqdm(train_loader):
    img,target = img.to(device),target.to(device)
    output = model(img)
    loss = criterion(output,target)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_loss += loss.item()
  train_loss /= len(train_loader)

  model.eval()
  test_preds = []
  test_labels = []
  test_loss = 0.0
  with torch.inference_mode():
    for img, target in tqdm(test_loader):
      img,target = img.to(device),target.to(device)
      output = model(img)
      loss = criterion(output,target)
      
      test_loss += loss.item()
      probs = torch.sofmax(output,dim=1)
      test_preds.append(probs.cpu().numpy())
      test_labels.extend(target.cpu().numpy())
    
    val_preds = np.concatenate(test_preds, axis=0)
    val_labels = np.array(test_labels)
    val_auc = roc_auc_score(
      y_true=np.eye(len(classes))[val_labels],
      y_score=val_preds,
      average="macro",
      multi_class="ovr",
    )

    print(f"Train loss: {train_loss:.4f} | Test loss: {test_loss:.4f} | ")



    
  

  0%|          | 0/500 [00:00<?, ?it/s]